In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
import rasterio
from rasterio.features import shapes, rasterize

In [3]:
import geopandas as gpd
from shapely.geometry import shape

In [4]:
from PIL import Image

In [5]:
# -------------------------
# CONFIG
# -------------------------
raster_path = "data/el_harrach_georef.tif"
output_dir = "output/vect/poly"
os.makedirs(output_dir, exist_ok=True)

In [6]:
TARGET_CRS = "EPSG:3857"

In [7]:
# -------------------------
# LEGEND
# -------------------------
df = pd.read_csv("data/legend_class_geo.csv")
df = df[df.geometry == "polygon"]

In [8]:
def hex_to_rgb(h):
    h = h.lstrip("#")
    return int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)

In [9]:
rgb_to_class = {
    hex_to_rgb(row["hex"]): row["class"]
    for _, row in df.iterrows()
}

In [10]:
classes = list(rgb_to_class.values())

In [11]:
class_map = {
    (r << 16 | g << 8 | b): i
    for i, ((r, g, b), _) in enumerate(rgb_to_class.items())
}

In [12]:
# -------------------------
# READ RASTER
# -------------------------
with rasterio.open(raster_path) as src:
    img = src.read()[:3]
    transform = src.transform
    crs = src.crs

In [13]:
if str(crs) != TARGET_CRS:
    raise ValueError(f"Expected raster CRS {TARGET_CRS}, got {crs}")

In [14]:
img = np.transpose(img, (1, 2, 0)).astype(np.uint8)
h, w, _ = img.shape

In [15]:
# -------------------------
# ENCODE RGB -> CLASS INDEX
# -------------------------
flat = img.reshape(-1, 3)

In [16]:
rgb_int = (
    flat[:, 0].astype(np.int32) << 16 |
    flat[:, 1].astype(np.int32) << 8 |
    flat[:, 2].astype(np.int32)
)

In [17]:
label = np.full(rgb_int.shape, -1, dtype=np.int32)

In [18]:
for rgb_key, idx in class_map.items():
    label[rgb_int == rgb_key] = idx

In [19]:
label = label.reshape(h, w)

In [20]:
# -------------------------
# VECTORIZE
# -------------------------
results = {c: [] for c in classes}

In [21]:
for geom, val in shapes(label, mask=label != -1, transform=transform):
    val = int(val)
    if val == -1:
        continue
    results[classes[val]].append(shape(geom))

In [22]:
# -------------------------
# EXPORT GEOJSONS
# -------------------------
for class_name, geoms in tqdm(results.items()):
    if not geoms:
        continue

    gdf = gpd.GeoDataFrame(geometry=geoms, crs=crs)
    gdf["class"] = class_name

    gdf.to_file(os.path.join(output_dir, f"{class_name}.geojson"), driver="GeoJSON")

  0%|                                          | 0/7 [00:00<?, ?it/s]

 14%|████▊                             | 1/7 [00:06<00:40,  6.81s/it]

 29%|█████████▋                        | 2/7 [00:07<00:16,  3.37s/it]

 57%|███████████████████▍              | 4/7 [00:08<00:04,  1.44s/it]

 86%|█████████████████████████████▏    | 6/7 [00:08<00:00,  1.22it/s]

100%|██████████████████████████████████| 7/7 [00:11<00:00,  1.45s/it]

100%|██████████████████████████████████| 7/7 [00:11<00:00,  1.70s/it]

In [23]:
# -------------------------
# SIMPLE RED MASK PNG (NO ALPHA, NO BLENDING)
# -------------------------
for class_name, geoms in tqdm(results.items()):
    if not geoms:
        continue

    mask = rasterize(
        [(geom, 1) for geom in geoms],
        out_shape=(h, w),
        transform=transform,
        fill=0,
        dtype=np.uint8
    )

    out = np.zeros((h, w, 3), dtype=np.uint8)
    out[mask == 1] = [255, 0, 0]

    Image.fromarray(out).save(
        os.path.join(output_dir, f"{class_name}.png")
    )

  0%|                                          | 0/7 [00:00<?, ?it/s]

 14%|████▊                             | 1/7 [00:10<01:05, 10.90s/it]

 29%|█████████▋                        | 2/7 [00:13<00:29,  5.84s/it]

 43%|██████████████▌                   | 3/7 [00:14<00:14,  3.62s/it]

 57%|███████████████████▍              | 4/7 [00:16<00:08,  2.95s/it]

 71%|████████████████████████▎         | 5/7 [00:16<00:04,  2.21s/it]

 86%|█████████████████████████████▏    | 6/7 [00:18<00:01,  1.99s/it]

100%|██████████████████████████████████| 7/7 [00:24<00:00,  3.16s/it]

100%|██████████████████████████████████| 7/7 [00:24<00:00,  3.45s/it]